In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from pyzbar.pyzbar import decode

def show_steps(images, titles, cols=4, figsize=(18, 9)):
    rows = int(np.ceil(len(images) / cols))
    plt.figure(figsize=figsize)
    for i, (img, title) in enumerate(zip(images, titles), 1):
        plt.subplot(rows, cols, i)
        if len(img.shape) == 2:
            plt.imshow(img, cmap="gray")
        else:
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title(title, fontsize=12)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

uploaded = files.upload()
filename = next(iter(uploaded))
file_bytes = uploaded[filename]
img_array = np.frombuffer(file_bytes, np.uint8)
img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
if img is None:
    raise ValueError("Could not read the uploaded file as an image. Use PNG/JPG/JPEG.")
original = img.copy()

gray = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5, 5), 0)
grad_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
morph_grad = cv2.morphologyEx(blur, cv2.MORPH_GRADIENT, grad_kernel)
_, otsu = cv2.threshold(morph_grad, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
close_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))
closed = cv2.morphologyEx(otsu, cv2.MORPH_CLOSE, close_kernel, iterations=2)

contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
blob_mask = np.zeros_like(gray)
contour_view = original.copy()
candidate_boxes = []
for c in sorted(contours, key=cv2.contourArea, reverse=True):
    area = cv2.contourArea(c)
    if area < 300:
        continue
    x, y, w, h = cv2.boundingRect(c)
    if w < 20 or h < 20:
        continue
    candidate_boxes.append((x, y, w, h))
    cv2.drawContours(blob_mask, [c], -1, 255, thickness=cv2.FILLED)
    cv2.rectangle(contour_view, (x, y), (x + w, y + h), (0, 255, 0), 2)

decoded_view = original.copy()
decoded_items = []
seen = set()

def add_decoded_result(obj, x_offset=0, y_offset=0):
    text = obj.data.decode("utf-8", errors="replace")
    key = (obj.type, text)
    if key in seen:
        return
    seen.add(key)
    x, y, w, h = obj.rect
    x += x_offset
    y += y_offset
    cv2.rectangle(decoded_view, (x, y), (x + w, y + h), (255, 0, 255), 3)
    cv2.putText(decoded_view, f"{obj.type}: {text}", (x, max(y - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 255), 2)
    decoded_items.append((obj.type, text))

full_results = decode(gray)
for obj in full_results:
    add_decoded_result(obj)

if not decoded_items:
    for (x, y, w, h) in candidate_boxes:
        pad = 10
        x1, y1 = max(0, x - pad), max(0, y - pad)
        x2, y2 = min(gray.shape[1], x + w + pad), min(gray.shape[0], y + h + pad)
        roi = gray[y1:y2, x1:x2]
        for obj in decode(roi):
            add_decoded_result(obj, x_offset=x1, y_offset=y1)

images = [original, gray, blur, morph_grad, otsu, closed, blob_mask, decoded_view]
titles = ["0. Original", "1. Grayscale", "2. Gaussian Blur", "3. Morph. Gradient",
          "4. Otsu Threshold", "5. Morph. Close", "6. Contour Detect / Blobs", "7. Decode"]
show_steps(images, titles)

print("Decoded results:")
if decoded_items:
    for typ, txt in decoded_items:
        print(f"- {typ}: {txt}")
else:
    print("No barcode / QR code detected.")
